# Longitudinal Modelling


This section demonstrates how to compute the [z-diff score](https://elifesciences.org/articles/95823) [1] to quantify longitudinal change in neuroimaging data relative to healthy ageing trajectories. We use a large-scale pre-trained Bayesian Linear Regression normative model [2, 3] from our pre-estimated library fitted on cortical thickness measures from the Destrieux parcellation across the lifespan. The target dataset consists of longitudinal structural MRI data from patients and healthy controls, with cortical thickness extracted using FreeSurfer, also documented in the original publication [1]. Rather than re-estimating the normative model on the new dataset, we leverage the pre-trained model directly by exploiting the fact that site-specific biases cancel out in the longitudinal difference, and compute z-diff scores that capture individual-level deviations from the expected healthy trajectory. 

-----
** References
- [1] Rehak Buckova, Barbora, et al. "Using normative models pre-trained on cross-sectional data to evaluate intra-individual longitudinal changes in neuroimaging data." Elife 13 (2025): RP95823.
- [2] Fraza, Charlotte J., et al. "Warped Bayesian linear regression for normative modelling of big data." NeuroImage 245 (2021): 118715.
- [3] Rutherford, Saige, et al. "Charting brain growth and aging at high spatial precision." elife 11 (2022): e72904.


## Setting up the environment

-----

<details>
<summary><b>System requirements</b></summary>

To follow this guide, you will need:
- Either a Mac, Linux, or Windows machine with WSL (Windows Subsystem for Linux) installed.
- About ___ GB of free disk space (for the data and the environment)
- About ___ GB of RAM (more is better, but not required)
- A decent processor (we recommend a multi-core CPU)
- A working internet connection (for installing the packages and downloading the data)

For Windows users with WSL, everything that follows will be done in the WSL terminal. You will have to follow the installation instructions for Linux/Mac. 

</details>

-----

<details>
<summary><b>Setting up Anaconda</b></summary>

Anaconda is a free and open-source distribution of the Python and R programming languages for scientific computing, that aims to simplify package management and deployment. We highly recommend using Anaconda (or an alternative like miniconda or venv) to manage your python environment. We will use Anaconda in this tutorial.

Follow the instructions on the anaconda website: https://www.anaconda.com/docs/getting-started/anaconda/install

Verify the installation by running the following command in your terminal:

```bash
conda --version
```

If you see the version number, you are good to go.

</details>

-----

<details>
<summary><b>Creating an environment and install the PCNtoolkit</b></summary>

Choose an environment name, here we will use `ptk` as a placeholder. We install pcntoolkit version 1.2.0.post1. We advise you to download this specific version as it is the version that this notebook was tested with. We cannot guarrantee that older or newer versions of pcntoolkit will work with this notebook.

```bash
conda create -n ptk python=3.12
conda activate ptk
pip install pcntoolkit==1.3.0
conda install -c conda-forge jupyter ipykernel ipywidgets
```

Now to verify that the installation is successful, run the following code:

```bash
python -c "import pcntoolkit as pcn; print(pcn.__version__)" 
```

This should print `1.3.0` without any errors.

</details>

-----

## Data Selection 🥦 

-----
In general, for z-diff scores to work appropriately, it is necessary to:
- access the pre-trained [BLR models](https://www.sciencedirect.com/science/article/pii/S1053811921009873) [1]
- have a subset of longitudinal healthy controls using the same scanner as the cohort of interest
- longitudinal cohort of interest

Data used in this tutorial were a part of the original [z-diff publication](https://elifesciences.org/articles/95823#metrics) [2].
For each participant, they contain individual age, sex, group (as in control/patient), visit (first or second), and [freesurfer](https://surfer.nmr.mgh.harvard.edu/) features, preprocessed with the `recon-all` command (version 7.2.). The mean cortical thickness values were extracted for the Destrieux parcellation [3].

Note that data was already prepared, specifically:
- sex has been coded in a manner compatible with the PCN toolkit 
- there are no missing values in the dataset
- each subject has two available visits

-----
** References
- [1] Fraza, Charlotte J., et al. "Warped Bayesian linear regression for normative modelling of big data." NeuroImage 245 (2021): 118715.
- [2] Rehak Buckova, Barbora, et al. "Using normative models pre-trained on cross-sectional data to evaluate intra-individual longitudinal changes in neuroimaging data." Elife 13 (2025): RP95823.
- [3] Destrieux, Christophe, et al. "Automatic parcellation of human cortical gyri and sulci using standard anatomical nomenclature." Neuroimage 53.1 (2010): 1-15.




In [14]:
# import necessary packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# load the longitudinanal data
try:
    df = pd.read_csv("../../data/LNM_data.csv")
except FileNotFoundError:
    print("File not found")

df.describe()


,age,lh_G&S_frontomargin_thickness,lh_G&S_occipital_inf_thickness,lh_G&S_paracentral_thickness,lh_G&S_subcentral_thickness,lh_G&S_transv_frontopol_thickness,lh_G&S_cingul-Ant_thickness,lh_G&S_cingul-Mid-Ant_thickness,lh_G&S_cingul-Mid-Post_thickness,lh_G_cingul-Post-dorsal_thickness,...,Right-Accumbens-area,Right-VentralDC,Right-vessel,Right-choroid-plexus,SubCortGrayVol,TotalGrayVol,SupraTentorialVol,SupraTentorialVolNotVent,EstimatedTotalIntraCranialVol,visit
count,330.000000,330.000000,330.000000,330.000000,330.000000,330.000000,330.000000,330.000000,330.000000,330.000000,...,330.000000,330.000000,330.000000,330.000000,330.000000,330.000000,3.300000e+02,3.300000e+02,3.300000e+02,330.000000
mean,29.351506,2.324761,2.393894,2.520776,2.671158,2.509967,2.665530,2.606042,2.534994,2.670512,...,631.609394,4395.611818,84.349394,1027.436970,63135.810510,682914.333802,1.031154e+06,1.009945e+06,1.599066e+06,1.500000
std,7.711299,0.122534,0.136400,0.129510,0.129045,0.124676,0.104851,0.135919,0.112216,0.133070,...,85.994449,420.457935,30.076455,206.075085,5110.498663,64636.677509,1.052995e+05,1.029850e+05,1.689117e+05,0.500759
min,18.052000,1.996000,1.978000,2.145000,2.402000,2.099000,2.215000,2.255000,2.239000,2.299000,...,432.100000,3368.800000,19.500000,514.800000,50837.744223,519746.966112,7.525749e+05,7.386196e+05,1.262103e+06,1.000000
25%,22.683000,2.249000,2.297000,2.443000,2.578000,2.433250,2.590500,2.519000,2.453500,2.580250,...,571.425000,4052.300000,64.450000,883.450000,59574.811589,636965.529664,9.484221e+05,9.313207e+05,1.460046e+06,1.000000
50%,28.370000,2.315500,2.396500,2.529500,2.663500,2.517000,2.665000,2.606000,2.527500,2.657500,...,631.200000,4390.100000,80.550000,994.900000,63126.405034,679366.465044,1.019905e+06,1.000600e+06,1.603637e+06,1.500000
75%,34.250000,2.404000,2.489750,2.605500,2.761750,2.586750,2.730000,2.693750,2.615000,2.755000,...,684.350000,4689.425000,100.200000,1172.700000,66715.899980,724746.167021,1.100884e+06,1.079135e+06,1.735659e+06,2.000000
max,54.933000,2.694000,2.790000,2.867000,3.019000,2.856000,3.014000,3.066000,2.877000,3.041000,...,971.200000,5494.400000,197.100000,1608.300000,76924.951824,863301.299368,1.308998e+06,1.291249e+06,2.016119e+06,2.000000


## Data preparation 🔪
----

### Download and unzip the pretrained normative models

In this stage, it is necessary to download the pre-trained normative models. For a detailed discussion of how to download and set up these models, refer to the notebook `4.1_model_transfer_lifespan_reference.ipynb`. Here, we will use `BLRw_ct_DES_lifespan_67K_89sites` as this parcellation is consistent with the one in our data. We download and extract the model into a separate directory.

In [15]:
import requests, zipfile, os

# Define the WebDAV URL and authentication credentials for Surfdrive to pull the model files from
BASE_URL    = "https://surfdrive.surf.nl/public.php/webdav/zip/"
SHARE_TOKEN = "Mb6mZyFmJeCaPcZ"
MODEL_NAME  = "BLRw_ct_DES_lifespan_67K_89sites"

data_dir  = os.path.abspath("../braincharts/data/")
model_dir = os.path.join(data_dir, MODEL_NAME)
zip_path  = os.path.join(data_dir, MODEL_NAME + ".zip")

os.makedirs(data_dir, exist_ok=True)

# download if not already on disk
if not os.path.exists(zip_path):
    print(f"Downloading {MODEL_NAME}...")
    r = requests.get(BASE_URL + MODEL_NAME + ".zip", auth=(SHARE_TOKEN, ""), stream=True)
    with open(zip_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)
    print("Done.")
else:
    print(f"{MODEL_NAME} already downloaded, skipping.")

# unzip if not already extracted
if not os.path.exists(model_dir):
    print("Unzipping...")
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(model_dir)
    print("Done.")
else:
    print(f"{MODEL_NAME} already extracted, skipping.")

BLRw_ct_DES_lifespan_67K_89sites already downloaded, skipping.
BLRw_ct_DES_lifespan_67K_89sites already extracted, skipping.


### Load the pretrained model
Next, we load the pretrained model and disable saving of results, plots, and the model itself, as we are only interested in the predictions.


In [16]:
import pcntoolkit as ptk

# create the directory for the results
results_dir = os.path.abspath("../../braincharts/results/")
os.makedirs(results_dir, exist_ok=True)

# load the pretrained normative model
model = ptk.NormativeModel.load(model_dir)
model.save_dir = results_dir
model.savemodel = False
model.saveresults = False
model.saveplots = False

c:\Users\kontsi\AppData\Local\anaconda3\envs\ptk-pu25\Lib\site-packages\pcntoolkit\util\output.py:296: UserWarning: Process: 17208 - 2026-06-23 14:32:15 - This model was saved with PCNtoolkit v1.1.1, but you are running v1.2.0.post1. Loading this model in v1.2.0.post1...
  warnings.warn(message, category)
c:\Users\kontsi\AppData\Local\anaconda3\envs\ptk-pu25\Lib\site-packages\pcntoolkit\util\output.py:296: UserWarning: Process: 17208 - 2026-06-23 14:32:15 - This model was saved with PCNtoolkit v1.1.2, but you are running v1.2.0.post1. Loading this model in v1.2.0.post1...
  warnings.warn(message, category)
c:\Users\kontsi\AppData\Local\anaconda3\envs\ptk-pu25\Lib\site-packages\pcntoolkit\util\output.py:296: UserWarning: Process: 17208 - 2026-06-23 14:32:16 - This model was saved with PCNtoolkit v1.1.2, but you are running v1.2.0.post1. Loading this model in v1.2.0.post1...
  warnings.warn(message, category)
c:\Users\kontsi\AppData\Local\anaconda3\envs\ptk-pu25\Lib\site-packages\pcntool

### Define covariates and response variables

We define the covariates, batch effects, and response variables. Only variables that are present in both the data and the pretrained model are retained as response variables.


In [17]:
# use the model's covariates and batch effects directly
subject_id = "sub_id"
covariates = model.covariates
batch_effects = list(model.unique_batch_effects.keys())
other_vars = ["visit", "group"]
non_respvars = [subject_id] + covariates + batch_effects + other_vars

# only keep response variables present in both the data and the model
response_vars = list(filter(lambda x: x not in non_respvars, df.columns))
response_vars = list(filter(lambda x: df[x].var() > 0, response_vars))
response_vars = list(filter(lambda x: df[x].dtype != "object", response_vars))
response_vars = list(set(model.response_vars) & set(response_vars))

print(f"Covariates: {covariates}")
print(f"Batch effects: {batch_effects}")
print(f"Number of response vars: {len(response_vars)}")
print(f"Pretrained sites: {list(model.unique_batch_effects['site'])[:5]} ... ({len(model.unique_batch_effects['site'])} total)")

Covariates: ['age']
Batch effects: ['site', 'sex']
Number of response vars: 148
Pretrained sites: ['ABCD_01', 'ABCD_02', 'ABCD_03', 'ABCD_04', 'ABCD_05'] ... (89 total)


### Prepare NormData objects

We split the data across groups and visits into separate `NormData` objects. This is not strictly necessary, however, it prevents accidental mistakes in further analyses and has a didactic purpose.

We also split the controls into an **adaptation set** (20%) used to transfer the pretrained model to the new site, and a **test set** (80%) used to estimate the healthy between-visit variance $2\sigma^2(1-\rho)$.


In [18]:
from sklearn.model_selection import train_test_split

# split controls into adaptation (20%) and test (80%) at subject level
ctrl_subjects = df[df['group'] == 'control']['sub_id'].unique().to_numpy()
adapt_subjects, test_ctrl_subjects = train_test_split(ctrl_subjects, test_size=0.8, random_state=42)

def make_normdata(dataframe, name):
    """Create a NormData object from a dataframe."""
    return ptk.NormData.from_dataframe(
        name=name,
        dataframe=dataframe,
        covariates=covariates,
        batch_effects=batch_effects,
        response_vars=response_vars,
        remove_Nan=True,
        remove_outliers=True,
        z_threshold=10
    )

# adaptation data: 20% of controls at V1 only
adapt = make_normdata(
    df[(df['group'] == 'control') &
       (df['visit'] == 1) &
       (df['sub_id'].isin(adapt_subjects))],
    'adapt'
)

# test controls: 80% of controls at both visits
con_v1 = make_normdata(
    df[(df['group'] == 'control') &
       (df['visit'] == 1) &
       (df['sub_id'].isin(test_ctrl_subjects))],
    'con_v1'
)
con_v2 = make_normdata(
    df[(df['group'] == 'control') &
       (df['visit'] == 2) &
       (df['sub_id'].isin(test_ctrl_subjects))],
    'con_v2'
)

# patients at both visits
pat_v1 = make_normdata(df[(df['group'] == 'patient') & (df['visit'] == 1)], 'pat_v1')
pat_v2 = make_normdata(df[(df['group'] == 'patient') & (df['visit'] == 2)], 'pat_v2')

print(f"Adaptation controls:  {adapt.dims['observations']} subjects")
print(f"Test controls V1/V2:  {con_v1.dims['observations']} / {con_v2.dims['observations']} subjects")
print(f"Patients V1/V2:       {pat_v1.dims['observations']} / {pat_v2.dims['observations']} subjects")

Process: 17208 - 2026-06-23 14:32:16 - Removed 0 NANs
Process: 17208 - 2026-06-23 14:32:16 - Removed 0 outliers
Process: 17208 - 2026-06-23 14:32:16 - Dataset "adapt" created.
    - 13 observations
    - 13 unique subjects
    - 1 covariates
    - 148 response variables
    - 2 batch effects:
    	site (1)
	sex (2)
    
Process: 17208 - 2026-06-23 14:32:16 - Removed 0 NANs
Process: 17208 - 2026-06-23 14:32:16 - Removed 0 outliers
Process: 17208 - 2026-06-23 14:32:16 - Dataset "con_v1" created.
    - 54 observations
    - 54 unique subjects
    - 1 covariates
    - 148 response variables
    - 2 batch effects:
    	site (1)
	sex (2)
    
Process: 17208 - 2026-06-23 14:32:16 - Removed 0 NANs
Process: 17208 - 2026-06-23 14:32:16 - Removed 0 outliers
Process: 17208 - 2026-06-23 14:32:16 - Dataset "con_v2" created.
    - 54 observations
    - 54 unique subjects
    - 1 covariates
    - 148 response variables
    - 2 batch effects:
    	site (1)
	sex (2)
    
Process: 17208 - 2026-06-23 14:3

C:\Users\kontsi\AppData\Local\Temp\ipykernel_17208\1949735827.py:46: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print(f"Adaptation controls:  {adapt.dims['observations']} subjects")
C:\Users\kontsi\AppData\Local\Temp\ipykernel_17208\1949735827.py:47: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print(f"Test controls V1/V2:  {con_v1.dims['observations']} / {con_v2.dims['observations']} subjects")
C:\Users\kontsi\AppData\Local\Temp\ipykernel_17208\1949735827.py:48: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to b

## Modelling 🥘
----


### Transfer and predict

We use the `transfer_predict` function to adapt the pretrained model to the new site using the adaptation controls, and then predict on the patient and control data. The same transferred model is used for predictions at both visits — this is important because the site-specific correction is identical at V1 and V2, and therefore cancels out in the longitudinal difference.

> ⚠️ **Note:** We use `transfer_predict` rather than `predict` directly, because the pretrained model was trained on 89 different sites and the new site must be explicitly adapted to. 

In [19]:
# transfer the pretrained model to the new site using adaptation controls (V1 only)
# and predict on patients at V1
transferred = model.transfer_predict(adapt, pat_v1)
transferred.save_dir = results_dir
transferred.savemodel = False
transferred.saveresults = False
transferred.saveplots = False

# predict on remaining datasets using the transferred model
transferred.predict(pat_v2)
transferred.predict(con_v1)
transferred.predict(con_v2)

Process: 17208 - 2026-06-23 14:32:16 - Transferring models on 148 response variables.
Process: 17208 - 2026-06-23 14:32:16 - Transferring model for lh_S_circular_insula_sup_thickness.
Process: 17208 - 2026-06-23 14:32:16 - Transferring model for rh_G_front_inf-Opercular_thickness.
Process: 17208 - 2026-06-23 14:32:16 - Transferring model for rh_G_temp_sup-Lateral_thickness.
Process: 17208 - 2026-06-23 14:32:16 - Transferring model for rh_S_suborbital_thickness.
Process: 17208 - 2026-06-23 14:32:16 - Transferring model for lh_S_temporal_sup_thickness.
Process: 17208 - 2026-06-23 14:32:16 - Transferring model for rh_S_central_thickness.
Process: 17208 - 2026-06-23 14:32:16 - Transferring model for rh_G_subcallosal_thickness.
Process: 17208 - 2026-06-23 14:32:16 - Transferring model for lh_G_temp_sup-Lateral_thickness.
Process: 17208 - 2026-06-23 14:32:16 - Transferring model for lh_G_cuneus_thickness.
Process: 17208 - 2026-06-23 14:32:16 - Transferring model for lh_S_central_thickness.
P

c:\Users\kontsi\AppData\Local\anaconda3\envs\ptk-pu25\Lib\site-packages\pcntoolkit\util\plotter.py:121: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  centile_df[rv] = 1e-6
c:\Users\kontsi\AppData\Local\anaconda3\envs\ptk-pu25\Lib\site-packages\pcntoolkit\util\plotter.py:121: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  centile_df[rv] = 1e-6
c:\Users\kontsi\AppData\Local\anaconda3\envs\ptk-pu25\Lib\site-packages\pcntoolkit\util\plotter.py:121: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of cal

Process: 17208 - 2026-06-23 14:33:22 - Dataset "centile" created.
    - 150 observations
    - 150 unique subjects
    - 1 covariates
    - 148 response variables
    - 2 batch effects:
    	site (1)
	sex (1)
    
Process: 17208 - 2026-06-23 14:33:22 - Computing centiles for 148 response variables.
Process: 17208 - 2026-06-23 14:33:22 - Computing centiles for lh_S_circular_insula_sup_thickness.
Process: 17208 - 2026-06-23 14:33:22 - Computing centiles for rh_G_front_inf-Opercular_thickness.
Process: 17208 - 2026-06-23 14:33:22 - Computing centiles for rh_G_temp_sup-Lateral_thickness.
Process: 17208 - 2026-06-23 14:33:22 - Computing centiles for rh_S_suborbital_thickness.
Process: 17208 - 2026-06-23 14:33:22 - Computing centiles for lh_S_temporal_sup_thickness.
Process: 17208 - 2026-06-23 14:33:22 - Computing centiles for rh_S_central_thickness.
Process: 17208 - 2026-06-23 14:33:22 - Computing centiles for rh_G_subcallosal_thickness.
Process: 17208 - 2026-06-23 14:33:22 - Computing cent

c:\Users\kontsi\AppData\Local\anaconda3\envs\ptk-pu25\Lib\site-packages\pcntoolkit\util\plotter.py:121: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  centile_df[rv] = 1e-6
c:\Users\kontsi\AppData\Local\anaconda3\envs\ptk-pu25\Lib\site-packages\pcntoolkit\util\plotter.py:121: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  centile_df[rv] = 1e-6
c:\Users\kontsi\AppData\Local\anaconda3\envs\ptk-pu25\Lib\site-packages\pcntoolkit\util\plotter.py:121: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of cal

Process: 17208 - 2026-06-23 14:35:12 - Dataset "centile" created.
    - 150 observations
    - 150 unique subjects
    - 1 covariates
    - 148 response variables
    - 2 batch effects:
    	site (1)
	sex (1)
    
Process: 17208 - 2026-06-23 14:35:12 - Computing centiles for 148 response variables.
Process: 17208 - 2026-06-23 14:35:12 - Computing centiles for lh_S_circular_insula_sup_thickness.
Process: 17208 - 2026-06-23 14:35:12 - Computing centiles for rh_G_front_inf-Opercular_thickness.
Process: 17208 - 2026-06-23 14:35:12 - Computing centiles for rh_G_temp_sup-Lateral_thickness.
Process: 17208 - 2026-06-23 14:35:12 - Computing centiles for rh_S_suborbital_thickness.
Process: 17208 - 2026-06-23 14:35:12 - Computing centiles for lh_S_temporal_sup_thickness.
Process: 17208 - 2026-06-23 14:35:12 - Computing centiles for rh_S_central_thickness.
Process: 17208 - 2026-06-23 14:35:12 - Computing centiles for rh_G_subcallosal_thickness.
Process: 17208 - 2026-06-23 14:35:12 - Computing cent

<xarray.NormData> Size: 680kB
Dimensions:            (observations: 54, response_vars: 148, covariates: 1,
                        batch_effect_dims: 2, centile: 5, statistic: 13)
Coordinates:
  * observations       (observations) int64 432B 0 1 2 3 4 5 ... 49 50 51 52 53
  * response_vars      (response_vars) <U34 20kB 'lh_S_circular_insula_sup_th...
  * covariates         (covariates) <U3 12B 'age'
  * batch_effect_dims  (batch_effect_dims) <U4 32B 'site' 'sex'
  * centile            (centile) float64 40B 0.05 0.25 0.5 0.75 0.95
  * statistic          (statistic) <U8 416B 'EXPV' 'Kurtosis' ... 'Skewness'
Data variables:
    subject_ids        (observations) int64 432B 0 1 2 3 4 5 ... 49 50 51 52 53
    Y                  (observations, response_vars) float64 64kB 2.52 ... 2.606
    X                  (observations, covariates) float64 432B 40.96 ... 23.73
    batch_effects      (observations, batch_effect_dims) <U7 3kB 'NIMH-CZ' .....
    Z                  (observations, response_vars) float64 64kB -2.112 ... ...
    centiles           (centile, observations, response_vars) float64 320kB 2...
    baseline_logp      (observations, response_vars) float64 64kB -0.6924 ......
    logp               (observations, response_vars) float64 64kB -2.48 ... -...
    Yhat               (observations, response_vars) float64 64kB 2.681 ... 2...
    statistics         (response_vars, statistic) float64 15kB 0.1966 ... -0....
Attributes:
    real_ids:                       False
    is_scaled:                      False
    name:                           con_v2
    unique_batch_effects:           {np.str_('site'): ['NIMH-CZ'], np.str_('s...
    batch_effect_counts:            defaultdict(<function NormData.register_b...
    covariate_ranges:               {np.str_('age'): {'min': 19.689, 'max': 5...
    batch_effect_covariate_ranges:  {np.str_('site'): {'NIMH-CZ': {np.str_('a...

### Transform predictions to warped space

The PCN toolkit returns predictions in the original (native) space rather than the warped space. However, the z-diff score must be computed in the warped space. We therefore apply the sinh-arcsinh warp to both the observed values $y$ and the predicted values $\hat{y}$ to recover the warped residuals $\varphi(y) - \varphi(\hat{y})$, which form the numerator of the z-diff equation.


In [20]:
def get_warped_residual(norm_data, fitted_model, responsevar):
    """Compute the residual in warped space: φ(y) - φ(ŷ)."""
    blr = fitted_model[responsevar]
    y = norm_data.Y.sel(response_vars=responsevar).values
    yhat = norm_data.Yhat.sel(response_vars=responsevar).values

    # apply sinh-arcsinh warp with fitted parameters gamma
    warped_y    = blr.warp.f(y, blr.gamma)
    warped_yhat = blr.warp.f(yhat, blr.gamma)

    return warped_y - warped_yhat  # φ(y) - φ(ŷ)

### Z-diff score computation

The z-diff score quantifies the longitudinal change in a brain measure relative to what would be expected based on healthy ageing, normalized by the uncertainty in that prediction:

$$z\text{-diff} = \frac{[\varphi(y^{(2)})-\varphi(y^{(1)})] - \mathbf{\bar{w}}^T[\phi(\mathbf{x}^{(2)})-\phi(\mathbf{x}^{(1)})]}{\sqrt{[\phi(\mathbf{x}^{(2)})-\phi(\mathbf{x}^{(1)})]^T\mathbf{A}^{-1}[\phi(\mathbf{x}^{(2)})-\phi(\mathbf{x}^{(1)})]+2\sigma^2(1-\rho)}}$$

The denominator has two components:
- **Epistemic uncertainty**: $[\phi(\mathbf{x}^{(2)})-\phi(\mathbf{x}^{(1)})]^T\mathbf{A}^{-1}[\phi(\mathbf{x}^{(2)})-\phi(\mathbf{x}^{(1)})]$ — reflects uncertainty in the model weights given the covariate change
- **Aleatoric uncertainty**: $2\sigma^2(1-\rho)$ — reflects the expected healthy between-visit variance, estimated empirically from longitudinal healthy controls

In this implementation, the epistemic uncertainty term is omitted from the denominator. This is because the posterior precision matrix $\mathbf{A}$ is estimated on the original 89-site training data and is not updated during transfer, making it unreliable for the new site. In practice, the epistemic term is negligible compared to the aleatoric term (on the order of $10^{-6}$ vs $10^{-2}$), so this simplification has minimal impact on the results. The denominator therefore reduces to:

$$\text{denominator} = \sqrt{\widehat{2\sigma^2(1-\rho)}}$$

where $\widehat{2\sigma^2(1-\rho)}$ is estimated from the healthy controls as:

$$\widehat{2\sigma^2(1-\rho)} = \frac{1}{|C|}\sum_{k \in C} \left[\varphi(y_{k}^{(2)})-\varphi(y_{k}^{(1)}) - \mathbf{\bar{w}}^T[\phi(\mathbf{x}_{k}^{(2)})-\phi(\mathbf{x}_{k}^{(1)})]\right]^2$$


In [21]:
def compute_zdiff(con_v1, con_v2, pat_v1, pat_v2, fitted_model):
    """Compute longitudinal z-diff scores for patients."""
    # use only response variables present in all four datasets
    response_vars = list(
        set(pat_v1.response_vars.values) &
        set(pat_v2.response_vars.values) &
        set(con_v1.response_vars.values) &
        set(con_v2.response_vars.values)
    )

    zdiff_mat = np.zeros((pat_v1.dims['observations'], len(response_vars)))

    for i, responsevar in enumerate(response_vars):

        # warped residuals φ(y) - φ(ŷ) for controls and patients
        res_cv1 = get_warped_residual(con_v1, fitted_model, responsevar)
        res_cv2 = get_warped_residual(con_v2, fitted_model, responsevar)
        res_pv1 = get_warped_residual(pat_v1, fitted_model, responsevar)
        res_pv2 = get_warped_residual(pat_v2, fitted_model, responsevar)

        # estimate 2σ²(1-ρ) from healthy controls:
        # average squared change in warped residuals across control subjects
        con_numerator = res_cv2 - res_cv1
        sigma2_rho    = max(np.mean(con_numerator**2), 1e-6)

        # patient z-diff
        pat_numerator = res_pv2 - res_pv1
        denominator   = np.sqrt(sigma2_rho)

        zdiff_mat[:, i] = pat_numerator / denominator

    return pd.DataFrame(zdiff_mat, index=pat_v1.observations.values, columns=response_vars)

In [22]:
zdiff = compute_zdiff(con_v1, con_v2, pat_v1, pat_v2, transferred)

print(f'z-diff shape: {zdiff.shape}')
print(f'z-diff range: {zdiff.min().min():.2f} to {zdiff.max().max():.2f}')
print(f'z-diff mean:  {zdiff.mean().mean():.2f}')

z-diff shape: (98, 148)
z-diff range: -12.64 to 5.30
z-diff mean:  0.02


C:\Users\kontsi\AppData\Local\Temp\ipykernel_17208\2196551217.py:11: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  zdiff_mat = np.zeros((pat_v1.dims['observations'], len(response_vars)))


## Evaluation 🍲
----

### Statistical testing

To identify brain regions with significant longitudinal change in patients, we apply a one-sample Wilcoxon signed-rank test to the z-diff scores for each response variable. This non-parametric test evaluates whether the median z-diff score across patients is significantly different from zero — i.e. whether patients show systematic deviation from the healthy ageing trajectory.

To control for multiple comparisons across all response variables, we apply Benjamini-Hochberg (BH) false discovery rate (FDR) correction. We also compute a signed p-value (`p_plot`) by multiplying the FDR-corrected p-value by the sign of the mean z-diff, which allows us to distinguish regions showing accelerated thinning (positive) from those showing decelerated thinning (negative) when visualizing results on the brain surface.


In [24]:
!pip install statsmodels

   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   -------------- ------------------------- 3.4/9.5 MB 18.3 MB/s eta 0:00:01
   ----------------------------- ---------- 7.1/9.5 MB 18.2 MB/s eta 0:00:01
   ---------------------------------------- 9.5/9.5 MB 17.5 MB/s  0:00:00

   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [statsmodels]
   -----------------

In [25]:
import statsmodels.stats.multitest as multi
from scipy.stats import wilcoxon

results = []
for roi in zdiff.columns:
    vals = zdiff[roi].dropna()
    if len(vals) > 0:
        w, p = wilcoxon(vals, zero_method='zsplit')
        results.append({'roi': roi, 'W': w, 'p': p})

results_df = pd.DataFrame(results)
results_df['p_fdr'] = multi.multipletests(results_df['p'], method='fdr_bh')[1]
results_df['mean_zdiff'] = zdiff.mean().values
results_df['p_plot'] = results_df['p_fdr'] * np.sign(results_df['mean_zdiff'])

print(f'Significant ROIs (FDR < 0.05): {(results_df["p_fdr"] < 0.05).sum()}')
print(results_df[results_df['p_fdr'] < 0.05].sort_values('p_fdr'))

Significant ROIs (FDR < 0.05): 7
                             roi       W         p     p_fdr  mean_zdiff  \
64      rh_S_front_sup_thickness  1124.0  0.000004  0.000590    0.534475   
84   rh_G_front_middle_thickness  1301.0  0.000068  0.004999    0.642074   
76   rh_S_front_middle_thickness  1437.0  0.000460  0.022711    0.424135   
90      rh_G_front_sup_thickness  1467.0  0.000682  0.024500    0.439046   
125  lh_G_front_middle_thickness  1482.0  0.000828  0.024500    0.380451   
14      lh_S_front_sup_thickness  1541.0  0.001723  0.036422    0.398215   
134  lh_S_front_middle_thickness  1532.0  0.001545  0.036422    0.335229   

       p_plot  
64   0.000590  
84   0.004999  
76   0.022711  
90   0.024500  
125  0.024500  
14   0.036422  
134  0.036422  


### Visualisation of longitudinal trajectories

To aid interpretation of the z-diff results, we visualise the longitudinal trajectories of individual patients for each significant region. Each subject is represented by two dots (V1 and V2) connected by a line, showing the direction and magnitude of change over time. Subjects whose z-diff score exceeds the threshold of $|z\text{-diff}| > 1.96$ (corresponding to a two-tailed $p < 0.05$ under a standard normal distribution) are highlighted in red, while non-significant subjects are shown in blue.

Use the dropdown menu to switch between significant regions.


In [26]:
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

sig_rois = results_df[results_df['p_fdr'] < 0.05].sort_values('p_fdr')['roi'].tolist()

def plot_roi(roi):
    sig_subjects = set(pat_v1.subject_ids.values[np.abs(zdiff[roi]) > 1.96])
    
    age_v1 = pd.Series(pat_v1.X.sel(covariates='age').values, index=pat_v1.subject_ids.values)
    age_v2 = pd.Series(pat_v2.X.sel(covariates='age').values, index=pat_v2.subject_ids.values)
    y_v1 = pd.Series(pat_v1.Y.sel(response_vars=roi).values, index=pat_v1.subject_ids.values)
    y_v2 = pd.Series(pat_v2.Y.sel(response_vars=roi).values, index=pat_v2.subject_ids.values)

    shared = sorted(set(age_v1.index) & set(age_v2.index))

    fig, ax = plt.subplots(figsize=(10, 6))

    for subj in shared:
        color = 'tomato' if subj in sig_subjects else 'steelblue'
        alpha = 0.8 if subj in sig_subjects else 0.3
        ax.plot([age_v1[subj], age_v2[subj]], [y_v1[subj], y_v2[subj]],
                color=color, alpha=alpha, linewidth=0.8)
        ax.scatter([age_v1[subj], age_v2[subj]], [y_v1[subj], y_v2[subj]],
                   color=color, alpha=alpha, s=20, zorder=3)

    from matplotlib.lines import Line2D
    ax.legend(handles=[
        Line2D([0], [0], color='tomato',    marker='o', label=f'|z-diff| > 1.96 (n={len(sig_subjects)})'),
        Line2D([0], [0], color='steelblue', marker='o', label='Non-significant'),
    ])
    
    mean_zdiff = zdiff[roi].mean()
    p_fdr = results_df[results_df['roi'] == roi]['p_fdr'].values[0]
    ax.set_xlabel('Age')
    ax.set_ylabel(f'Y ({roi})')
    ax.set_title(f'{roi}\nmean z-diff={mean_zdiff:.2f}, FDR p={p_fdr:.4f}')
    plt.tight_layout()
    plt.show()

widgets.interact(
    plot_roi,
    roi=widgets.Dropdown(options=sig_rois, description='ROI:')
)

interactive(children=(Dropdown(description='ROI:', options=('rh_S_front_sup_thickness', 'rh_G_front_middle_thi…

<function __main__.plot_roi(roi)>